In [1]:
from google.colab import files

uploaded = files.upload()


Saving a-blood-analysis-lab-protocol.pdf to a-blood-analysis-lab-protocol.pdf
Saving CSI-clinical-practice-guidelines-for-dyslipidemia-_240413_141815.pdf to CSI-clinical-practice-guidelines-for-dyslipidemia-_240413_141815.pdf
Saving Guidelines-Made-Simple-Tool-2018-Cholesterol.pdf to Guidelines-Made-Simple-Tool-2018-Cholesterol.pdf
Saving sterling-accuris-pathology-sample-report-unlocked.pdf to sterling-accuris-pathology-sample-report-unlocked.pdf
Saving Hospital_Sample_Blood_Report.pdf to Hospital_Sample_Blood_Report.pdf
Saving Sample_Consolidated_Blood_Report.pdf to Sample_Consolidated_Blood_Report.pdf
Saving CBC-test-report-format-example-sample-template-Drlogy-lab-report.pdf to CBC-test-report-format-example-sample-template-Drlogy-lab-report.pdf


In [2]:
!pip install pdfplumber pytesseract pdf2image
!apt-get install -y tesseract-ocr


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 91.3 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [3]:
import os
import re
import pandas as pd
import pdfplumber
import pytesseract
from pdf2image import convert_from_path


In [4]:
def extract_text_from_pdf(pdf_path):
    text_data = ""

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    text_data += text + "\n"
                else:
                    img = page.to_image(resolution=300).original
                    text_data += pytesseract.image_to_string(img) + "\n"
    except:
        images = convert_from_path(pdf_path)
        for img in images:
            text_data += pytesseract.image_to_string(img) + "\n"

    return text_data


In [5]:
def extract_text_from_pdf(pdf_path):
    text_data = ""

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    text_data += text + "\n"
                else:
                    img = page.to_image(resolution=300).original
                    text_data += pytesseract.image_to_string(img) + "\n"
    except:
        images = convert_from_path(pdf_path)
        for img in images:
            text_data += pytesseract.image_to_string(img) + "\n"

    return text_data


In [6]:
pdf_files = list(uploaded.keys())
print("PDFs found:", pdf_files)

all_text = ""

for pdf in pdf_files:
    print("Processing:", pdf)
    all_text += extract_text_from_pdf(pdf)


PDFs found: ['a-blood-analysis-lab-protocol.pdf', 'CSI-clinical-practice-guidelines-for-dyslipidemia-_240413_141815.pdf', 'Guidelines-Made-Simple-Tool-2018-Cholesterol.pdf', 'sterling-accuris-pathology-sample-report-unlocked.pdf', 'Hospital_Sample_Blood_Report.pdf', 'Sample_Consolidated_Blood_Report.pdf', 'CBC-test-report-format-example-sample-template-Drlogy-lab-report.pdf']
Processing: a-blood-analysis-lab-protocol.pdf
Processing: CSI-clinical-practice-guidelines-for-dyslipidemia-_240413_141815.pdf
Processing: Guidelines-Made-Simple-Tool-2018-Cholesterol.pdf
Processing: sterling-accuris-pathology-sample-report-unlocked.pdf
Processing: Hospital_Sample_Blood_Report.pdf
Processing: Sample_Consolidated_Blood_Report.pdf
Processing: CBC-test-report-format-example-sample-template-Drlogy-lab-report.pdf


In [7]:
cleaned_text = all_text.lower()
cleaned_text = re.sub(r"\s+", " ", cleaned_text)

with open("combined_cleaned_text.txt", "w") as f:
    f.write(cleaned_text)

print("Text cleaned & saved")


Text cleaned & saved


In [8]:
patterns = {
    "hemoglobin": r"hemoglobin[^0-9]*([0-9.]+)",
    "rbc": r"rbc[^0-9]*([0-9.]+)",
    "wbc": r"wbc[^0-9]*([0-9,]+)",
    "platelets": r"platelet[^0-9]*([0-9,]+)",
    "glucose": r"glucose[^0-9]*([0-9.]+)",
    "cholesterol": r"cholesterol[^0-9]*([0-9.]+)"
}

extracted = {}

for key, pattern in patterns.items():
    values = re.findall(pattern, cleaned_text)
    extracted[key] = values[:20]  # limit for sanity

df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in extracted.items()]))
df.to_csv("extracted_parameters.csv", index=False)

df.head()


,hemoglobin,rbc,wbc,platelets,glucose,cholesterol
0,1,4.79,10570,150000,1,1
1,14.5,7,7,7,40,88
2,1,001,001,001,100,2005
3,1995,001,001,150000,157.07,19
4,2,5.2,9000,NaN,1,2023


In [9]:
for col in df.columns:
    df[col] = df[col].astype(str).str.replace(",", "")
    df[col] = pd.to_numeric(df[col], errors="coerce")


In [10]:
RANGES = {
    "hemoglobin": (5, 20),
    "rbc": (2, 8),
    "wbc": (2000, 30000),
    "platelets": (50000, 1000000),
    "glucose": (40, 400),
    "cholesterol": (50, 400)
}

def status(val, low, high):
    if pd.isna(val):
        return "missing"
    if val < low:
        return "low"
    if val > high:
        return "high"
    return "normal"

for col, (low, high) in RANGES.items():
    df[col + "_status"] = df[col].apply(lambda x: status(x, low, high))

df.to_csv("ms1_validated_standardized_parameters.csv", index=False)
df.head()


,hemoglobin,rbc,wbc,platelets,glucose,cholesterol,hemoglobin_status,rbc_status,wbc_status,platelets_status,glucose_status,cholesterol_status
0,1.0,4.79,10570.0,150000.0,1.00,1.0,low,normal,normal,normal,low,low
1,14.5,7.00,7.0,7.0,40.00,88.0,normal,normal,low,low,normal,normal
2,1.0,1.00,1.0,1.0,100.00,2005.0,low,low,low,low,normal,high
3,1995.0,1.00,1.0,150000.0,157.07,19.0,high,low,low,normal,normal,low
4,2.0,5.20,9000.0,NaN,1.00,2023.0,low,normal,normal,missing,low,high


In [11]:
def identify_patterns(row):
    patterns = []

    if row["hemoglobin_status"] == "low" and row["rbc_status"] == "low":
        patterns.append("possible_anemia")

    if row["glucose_status"] == "high":
        patterns.append("high_glucose")

    if row["cholesterol_status"] == "high":
        patterns.append("high_cholesterol")

    if row["wbc_status"] == "high":
        patterns.append("possible_infection")

    return ", ".join(patterns) if patterns else "no_significant_pattern"

df["identified_patterns"] = df.apply(identify_patterns, axis=1)
df.to_csv("ms2_pattern_analysis.csv", index=False)

df[["identified_patterns"]].head()


,identified_patterns
0,no_significant_pattern
1,no_significant_pattern
2,"possible_anemia, high_cholesterol"
3,no_significant_pattern
4,high_cholesterol


In [12]:
def generate_recommendation(row):
    recs = []

    if "possible_anemia" in row["identified_patterns"]:
        recs.append("Increase iron intake and consult physician")

    if "high_glucose" in row["identified_patterns"]:
        recs.append("Reduce sugar intake and monitor glucose")

    if "high_cholesterol" in row["identified_patterns"]:
        recs.append("Adopt low-fat diet and exercise")

    if "possible_infection" in row["identified_patterns"]:
        recs.append("Seek medical evaluation")

    return " | ".join(recs) if recs else "Maintain healthy lifestyle"

df["final_summary"] = df["identified_patterns"].apply(
    lambda x: "Abnormal pattern detected" if x != "no_significant_pattern"
    else "No abnormality detected"
)

df["recommendation"] = df.apply(generate_recommendation, axis=1)
df.to_csv("ms3_final_health_recommendations.csv", index=False)

df.head()


,hemoglobin,rbc,wbc,platelets,glucose,cholesterol,hemoglobin_status,rbc_status,wbc_status,platelets_status,glucose_status,cholesterol_status,identified_patterns,final_summary,recommendation
0,1.0,4.79,10570.0,150000.0,1.00,1.0,low,normal,normal,normal,low,low,no_significant_pattern,No abnormality detected,Maintain healthy lifestyle
1,14.5,7.00,7.0,7.0,40.00,88.0,normal,normal,low,low,normal,normal,no_significant_pattern,No abnormality detected,Maintain healthy lifestyle
2,1.0,1.00,1.0,1.0,100.00,2005.0,low,low,low,low,normal,high,"possible_anemia, high_cholesterol",Abnormal pattern detected,Increase iron intake and consult physician | A...
3,1995.0,1.00,1.0,150000.0,157.07,19.0,high,low,low,normal,normal,low,no_significant_pattern,No abnormality detected,Maintain healthy lifestyle
4,2.0,5.20,9000.0,NaN,1.00,2023.0,low,normal,normal,missing,low,high,high_cholesterol,Abnormal pattern detected,Adopt low-fat diet and exercise


In [13]:
from google.colab import files

files.download("ms1_validated_standardized_parameters.csv")
files.download("ms2_pattern_analysis.csv")
files.download("ms3_final_health_recommendations.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import pandas as pd

df = pd.read_csv("ms3_final_health_recommendations.csv")
df.head()


,hemoglobin,rbc,wbc,platelets,glucose,cholesterol,hemoglobin_status,rbc_status,wbc_status,platelets_status,glucose_status,cholesterol_status,identified_patterns,final_summary,recommendation
0,1.0,4.79,10570.0,150000.0,1.00,1.0,low,normal,normal,normal,low,low,no_significant_pattern,No abnormality detected,Maintain healthy lifestyle
1,14.5,7.00,7.0,7.0,40.00,88.0,normal,normal,low,low,normal,normal,no_significant_pattern,No abnormality detected,Maintain healthy lifestyle
2,1.0,1.00,1.0,1.0,100.00,2005.0,low,low,low,low,normal,high,"possible_anemia, high_cholesterol",Abnormal pattern detected,Increase iron intake and consult physician | A...
3,1995.0,1.00,1.0,150000.0,157.07,19.0,high,low,low,normal,normal,low,no_significant_pattern,No abnormality detected,Maintain healthy lifestyle
4,2.0,5.20,9000.0,NaN,1.00,2023.0,low,normal,normal,missing,low,high,high_cholesterol,Abnormal pattern detected,Adopt low-fat diet and exercise


In [15]:
def score_risk(row):
    score = 0

    if "possible_anemia" in row["identified_patterns"]:
        score += 2
    if "high_glucose" in row["identified_patterns"]:
        score += 2
    if "high_cholesterol" in row["identified_patterns"]:
        score += 2
    if "possible_infection" in row["identified_patterns"]:
        score += 1

    return score

df["risk_score"] = df.apply(score_risk, axis=1)
df[["identified_patterns", "risk_score"]].head()


,identified_patterns,risk_score
0,no_significant_pattern,0
1,no_significant_pattern,0
2,"possible_anemia, high_cholesterol",4
3,no_significant_pattern,0
4,high_cholesterol,2


In [16]:
def risk_category(score):
    if score >= 5:
        return "High Risk"
    if score >= 2:
        return "Moderate Risk"
    return "Low Risk"

df["risk_category"] = df["risk_score"].apply(risk_category)
df[["risk_score", "risk_category"]].value_counts()


,,count
risk_score,risk_category,
0,Low Risk,18
2,Moderate Risk,1
4,Moderate Risk,1


In [17]:
def synthesize_findings(row):
    findings = []

    if "possible_anemia" in row["identified_patterns"]:
        findings.append("Possible anemia indicators found")
    if "high_glucose" in row["identified_patterns"]:
        findings.append("High glucose risk detected")
    if "high_cholesterol" in row["identified_patterns"]:
        findings.append("High cholesterol risk detected")
    if "possible_infection" in row["identified_patterns"]:
        findings.append("Possible infection/inflammation detected")

    if not findings:
        findings.append("No major abnormalities detected")

    return "; ".join(findings)

df["synthesized_findings"] = df.apply(synthesize_findings, axis=1)
df[["identified_patterns", "synthesized_findings"]].head()


,identified_patterns,synthesized_findings
0,no_significant_pattern,No major abnormalities detected
1,no_significant_pattern,No major abnormalities detected
2,"possible_anemia, high_cholesterol",Possible anemia indicators found; High cholest...
3,no_significant_pattern,No major abnormalities detected
4,high_cholesterol,High cholesterol risk detected


In [18]:
def generate_report(row, patient_id):
    report = f"""
===============================
AI HEALTH DIAGNOSTIC REPORT
===============================

Patient ID: {patient_id}

Risk Category: {row['risk_category']}
Risk Score: {row['risk_score']}

Findings:
{row['synthesized_findings']}

Recommendation:
{row['recommendation']}

Disclaimer:
This report is AI-assisted and intended for preliminary assessment only.
Please consult a medical professional for diagnosis.
===============================
"""
    return report


In [19]:
reports = []

for i, row in df.iterrows():
    reports.append(generate_report(row, patient_id=i))

with open("ms4_final_health_reports.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(reports))

print("Saved ms4_final_health_reports.txt ✅")


Saved ms4_final_health_reports.txt ✅


In [20]:
df.to_csv("ms4_final_pipeline_output.csv", index=False)
print("Saved ms4_final_pipeline_output.csv ✅")


Saved ms4_final_pipeline_output.csv ✅


In [21]:
from google.colab import files

files.download("ms4_final_pipeline_output.csv")
files.download("ms4_final_health_reports.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>